# AACAgent — Evaluation Notebook

Valutazione su `annotation/eval_annotated.parquet` (1 760 righe).

Esegue due run separate per ogni frase: `input_type=clear` e `input_type=vague`,
a seconda del valore della colonna `split`. I tool MCP (`get_time`, `get_schedule`)
sono iniettati via `EvalContext` con i valori sintetici annotati nel parquet,
senza chiamate live.

**Output:** CSV con le sole scelte del modello (no metriche). Le metriche
(`hit`, `gold_in_candidates`, `overlap`) si calcolano in un notebook separato
incrociando col parquet.

**Colonne CSV principali:**
```
row_idx, input_type, turn_pos, concept_text,
called_get_time, called_get_schedule,
predicted_ids, plan_method, resolve_method, planner_concepts
```

In [ ]:
# ─── COLAB ONLY — skip on cluster ────────────────────────────────────────────
import subprocess, sys, os

subprocess.run(
    ["git", "clone", "https://github.com/lollopelle01/aac-mcp-agent.git",
     "/content/aac-mcp-agent"],
    check=True,
)

PROJECT_ROOT = "/content/aac-mcp-agent"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "app", "src"))
sys.path.insert(0, PROJECT_ROOT)

print("Repo cloned. CWD:", os.getcwd())

In [ ]:
# ─── COLAB ENV VARS — edit these before running ───────────────────────────────
# On the cluster every variable is injected by run_eval_cluster.sh — do not
# edit this cell for cluster runs.

import os

os.environ["NB_IS_COLAB"]          = "True"
os.environ["NB_MODELS"]            = "Qwen/Qwen2.5-3B-Instruct"
os.environ["NB_N_ROWS"]            = "100"   # 0 = tutte le 1760 frasi
os.environ["NB_LANG"]              = "en_eval"
os.environ["NB_MAX_NEW_TOKENS"]    = "512"
os.environ["NB_LOAD_8BIT"]         = "True"
os.environ["NB_MAX_RESULTS"]       = "0"     # 0 = usa default settings.py (25)
os.environ["NB_SEED"]              = "42"
os.environ["NB_SPLIT_FILTER"]      = "all"   # "clear"|"vague"|"both"|"all"
os.environ["NB_OUTPUT_CSV"]        = "/content/aac-mcp-agent/eval/results/eval_annotated_colab.csv"
os.environ["NB_ANNOTATED_PARQUET"] = "/content/aac-mcp-agent/annotation/eval_annotated.parquet"

In [ ]:
import os

# ── Environment detection ─────────────────────────────────────────────────────
_is_colab_env = os.environ.get("NB_IS_COLAB", "").lower()
if _is_colab_env == "false":
    IS_COLAB = False
elif _is_colab_env == "true":
    IS_COLAB = True
else:
    IS_COLAB = "google.colab" in str(globals().get("__builtins__", ""))

print(f"Running on: {'Colab/local' if IS_COLAB else 'Cluster'}")

# ── Core parameters ───────────────────────────────────────────────────────────
MODELS_RAW        = os.environ.get("NB_MODELS",            "Qwen/Qwen2.5-3B-Instruct")
N_ROWS_ENV        = os.environ.get("NB_N_ROWS",             "0")   # 0 = full dataset
SEED              = int(os.environ.get("NB_SEED",            "42"))
LOAD_8BIT_ENV     = os.environ.get("NB_LOAD_8BIT",           "False")
MAX_NEW_TOKENS    = int(os.environ.get("NB_MAX_NEW_TOKENS",   "512"))
LANG_CODE         = os.environ.get("NB_LANG",                "en_eval")
_max_results_env  = int(os.environ.get("NB_MAX_RESULTS",     "0"))
SPLIT_FILTER      = os.environ.get("NB_SPLIT_FILTER",        "all")
ANNOTATED_PARQUET = os.environ.get("NB_ANNOTATED_PARQUET",
                                   "annotation/eval_annotated.parquet")

# OUTPUT_CSV: cluster writes eval_hf.csv, Colab writes eval_annotated_colab.csv.
_default_csv = "eval/results/eval_annotated_colab.csv" if IS_COLAB else "eval/results/eval_annotated_hf.csv"
OUTPUT_CSV   = os.environ.get("NB_OUTPUT_CSV", _default_csv)

# On Colab, cap rows to keep execution time reasonable.
# Cap applies only when N_ROWS == 0 (no explicit limit set).
COLAB_SAMPLE = 200

# Parse compound types
MODELS    = MODELS_RAW.split()
N_ROWS    = int(N_ROWS_ENV)
LOAD_8BIT = LOAD_8BIT_ENV.lower() == "true"
HF_DEVICE = "auto"

print(f"Models            : {MODELS}")
print(f"N_rows            : {N_ROWS if N_ROWS > 0 else 'full dataset'}")
print(f"Seed              : {SEED}")
print(f"Load 8-bit        : {LOAD_8BIT}")
print(f"Max new tokens    : {MAX_NEW_TOKENS}")
print(f"Lang              : {LANG_CODE}")
print(f"Max results       : {_max_results_env if _max_results_env > 0 else 'default (settings.py)'}")
print(f"Split filter      : {SPLIT_FILTER}")
print(f"Annotated parquet : {ANNOTATED_PARQUET}")
print(f"Output CSV        : {OUTPUT_CSV}")

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip(
    "transformers>=4.40,<5.0",
    "accelerate>=0.29",
    "datasets>=2.19",
    "pandas>=2.0",
    "pyarrow>=14",
    "huggingface_hub>=0.22",
    "tqdm>=4.66",
    "sentencepiece>=0.1.99",
    "protobuf>=3.20",
    "spacy>=3.7",
    "fastmcp",
    "pydantic>=2.0",
    "httpx>=0.24",
    "python-dotenv",
)

if LOAD_8BIT:
    _pip("bitsandbytes>=0.43")

# spaCy model
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("spaCy model en_core_web_sm already present.")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

print("All packages ready.")

In [ ]:
from pathlib import Path

# Detect project root.
# Notebook lives at <project_root>/eval/eval.ipynb
_NB_DIR = Path().resolve()
if (_NB_DIR / "eval" / "eval.ipynb").exists():
    PROJECT_ROOT = _NB_DIR           # CWD is project root (cluster via papermill)
else:
    PROJECT_ROOT = _NB_DIR.parent    # CWD is eval/ (interactive run)

SRC = PROJECT_ROOT / "app" / "src"
APP = PROJECT_ROOT / "app"

for p in [str(SRC), str(APP)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Key paths ─────────────────────────────────────────────────────────────────
# Annotated dataset (R27): sentence-based with synthetic context columns
EVAL_PARQUET = (
    PROJECT_ROOT / ANNOTATED_PARQUET
    if not Path(ANNOTATED_PARQUET).is_absolute()
    else Path(ANNOTATED_PARQUET)
)

# Output CSV
_out = Path(OUTPUT_CSV)
if not _out.is_absolute():
    _out = PROJECT_ROOT / _out
_out.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = _out

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"SRC           : {SRC}")
print(f"Eval parquet  : {EVAL_PARQUET}  exists={EVAL_PARQUET.exists()}")
print(f"Output CSV    : {OUTPUT_CSV_PATH}")

In [ ]:
import ast
import csv
import logging
import time
from datetime import timedelta
from typing import Optional

import pandas as pd
import torch

# ── Logging ───────────────────────────────────────────────────────────────────
try:
    from logs.logging_config import setup_logging
    setup_logging()
except Exception:
    logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

# ── Project imports ───────────────────────────────────────────────────────────
from config import AGENT_MAX_RESULTS
from agent.agent import AACAgent, EvalContext
from agent.hf_agent import HFAACAgent
from agent.session import SessionMemory
from mcp_server.models import Pictogram, Keyword
from mcp_server.tools.arasaac import get_pictogram_metadata
import mcp_server.tools.arasaac as _arasaac_mod

# Use the frozen eval dataset for all ARASAAC lookups
_arasaac_mod.LANG = LANG_CODE  # type: ignore[attr-defined]

# Effective window size: env override (NB_MAX_RESULTS > 0) or settings default
EVAL_MAX_RESULTS = _max_results_env if _max_results_env > 0 else AGENT_MAX_RESULTS

print("Imports OK.")
print(f"CUDA available   : {torch.cuda.is_available()}")
print(f"EVAL_MAX_RESULTS : {EVAL_MAX_RESULTS}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)")

In [ ]:
# Output CSV columns — order determines column order in the file
# NON includere: hit, gold_id, candidate_ids, gold_in_candidates,
# overlap_level, n_candidates, window_len, synset_added, fresh_count.
# Tutto questo si calcola nel notebook metriche incrociando col parquet.
CSV_COLUMNS = [
    "row_idx",             # indice nel df campionato → chiave di join col parquet
    "input_type",          # "clear" | "vague"
    "turn_pos",            # 0-based nella sequenza multi-turn
    "concept_text",        # concept gold di questo turno (per join con concepts[turn_pos])
    "called_get_time",     # True se il modello ha chiamato get_time questo turno
    "called_get_schedule", # True se il modello ha chiamato get_schedule questo turno
    "predicted_ids",       # list[int] serializzata — finestra restituita dal modello
    "plan_method",         # "llm" | "fallback_spacy" | "fallback_empty"
    "resolve_method",      # "exact"|"lemma"|"hyphen"|"lemma_alt"|"token"|"none"
    "planner_concepts",    # list[str] serializzata — concetti generati dal planner
]

In [ ]:
print(f"Loading dataset from {EVAL_PARQUET} …")
df_full = pd.read_parquet(EVAL_PARQUET)
print(f"Full dataset : {len(df_full):,} rows × {df_full.shape[1]} cols")
print(f"Columns      : {list(df_full.columns)}")

# Deserialise `concepts` if stored as JSON string
if df_full["concepts"].dtype == object and isinstance(df_full["concepts"].iloc[0], str):
    df_full["concepts"] = df_full["concepts"].apply(ast.literal_eval)

# Deserialise `schedule` if stored as JSON string
if df_full["schedule"].dtype == object and isinstance(df_full["schedule"].iloc[0], str):
    df_full["schedule"] = df_full["schedule"].apply(ast.literal_eval)

# Sanity-check structure
assert set(["sentence", "concepts", "caregiver_clear", "caregiver_vague",
            "time_of_day", "event_time", "schedule", "split"]).issubset(df_full.columns), (
    f"Missing expected columns. Found: {list(df_full.columns)}"
)
assert df_full["concepts"].iloc[0][0].get("concept_text") is not None, (
    f"Unexpected concepts structure: {df_full['concepts'].iloc[0][0]}\n"
    "Expected: [{'concept_text': ..., 'gold_id': ..., 'candidate_ids': [...]}, ...]"
)
print(f"Sanity OK — example concept: {df_full['concepts'].iloc[0][0]}")
print(f"split distribution:\n{df_full['split'].value_counts()}")

# ── Sampling ──────────────────────────────────────────────────────────────────
if IS_COLAB:
    _cap = N_ROWS if N_ROWS > 0 else COLAB_SAMPLE
    df   = df_full.sample(min(_cap, len(df_full)), random_state=SEED).reset_index(drop=True)
    print(f"Colab sample : {len(df)} rows (cap={_cap}, seed={SEED})")
else:
    if N_ROWS > 0:
        df = df_full.sample(N_ROWS, random_state=SEED).reset_index(drop=True)
        print(f"Sampled      : {len(df)} rows (seed={SEED})")
    else:
        df = df_full
        print("Using full dataset (cluster mode)")

print(f"Effective rows: {len(df)}")
print(f"\nRow 0 sample:")
print(df.iloc[0].to_string())

In [ ]:
# ── Gold metadata cache ───────────────────────────────────────────────────────

_gold_cache: dict[int, dict] = {}

def get_gold_meta(pic_id: int) -> dict:
    """Fetch and cache pictogram metadata for a gold ID."""
    k = int(pic_id)
    if k not in _gold_cache:
        try:
            _gold_cache[k] = get_pictogram_metadata(pictogram_id=k, lang=LANG_CODE)
        except Exception:
            _gold_cache[k] = {}
    return _gold_cache[k]


def gold_as_pictogram(pic_id: int, concept: str) -> Pictogram:
    """Return a Pictogram for a gold ID (fallback to minimal stub if metadata missing)."""
    meta = get_gold_meta(pic_id)
    if meta:
        try:
            return Pictogram.model_validate(meta)
        except Exception:
            pass
    return Pictogram(id=int(pic_id), keywords=[Keyword(type=2, keyword=concept)])


# ── EvalContext builder ───────────────────────────────────────────────────────

def build_eval_ctx(row) -> EvalContext:
    """Costruisce EvalContext con i mock values dalla riga annotata.
    Viene usato SOLO al turn_pos==0. Ai turni successivi si passa
    EvalContext() vuoto — il contesto è già in sessione via memory."""
    mock_time = {
        "time_of_day": row["time_of_day"],
        "event_time":  str(row["event_time"]),
    }
    raw_sched = row["schedule"]
    if isinstance(raw_sched, str):
        raw_sched = ast.literal_eval(raw_sched)
    mock_schedule = raw_sched if isinstance(raw_sched, list) else []
    return EvalContext(mock_time=mock_time, mock_schedule=mock_schedule)


# ── Teacher forcing ───────────────────────────────────────────────────────────

def teacher_force(agent: AACAgent, gold_id: int, concept: str) -> None:
    """Replace the last memory turn with the gold pictogram.

    Simulates the caregiver manually finding and selecting the correct
    pictogram when the agent missed it, keeping the multi-turn sequence
    in a consistent state for subsequent turns.
    """
    if not agent.memory.turns:
        return
    last = agent.memory.turns[-1]
    for t in last.topics:
        agent.memory.topic_frequency[t] = max(0, agent.memory.topic_frequency.get(t, 0) - 1)
    gold_pic    = gold_as_pictogram(gold_id, concept)
    gold_topics = SessionMemory.extract_topics([gold_pic])
    last.pictograms = [gold_pic]
    last.topics     = gold_topics
    for t in gold_topics:
        agent.memory.topic_frequency[t] = agent.memory.topic_frequency.get(t, 0) + 1


# ── Incremental CSV writer ────────────────────────────────────────────────────

class IncrementalCSV:
    """Append-mode CSV writer with buffering. Creates header on first write."""

    def __init__(self, path: Path) -> None:
        self.path    = path
        self._is_new = not path.exists()
        self._buffer: list[dict] = []

    def add(self, rows: list[dict]) -> None:
        self._buffer.extend(rows)

    def flush(self) -> None:
        if not self._buffer:
            return
        mode = "w" if self._is_new else "a"
        with open(self.path, mode, newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
            if self._is_new:
                writer.writeheader()
                self._is_new = False
            writer.writerows(self._buffer)
        self._buffer.clear()


# ── Progress tracker ──────────────────────────────────────────────────────────

class ProgressTracker:
    """Tracks errors and ETA across all rows for one model."""

    def __init__(self, total: int) -> None:
        self.total    = total
        self.n_done   = 0
        self.n_errors = 0
        self._t0      = time.monotonic()

    def record(self, row_results: list[dict]) -> None:
        self.n_done += 1

    def record_error(self) -> None:
        self.n_done  += 1
        self.n_errors += 1

    def print_progress(self) -> None:
        elapsed = time.monotonic() - self._t0
        avg_s   = elapsed / max(self.n_done, 1)
        eta_str = str(timedelta(seconds=int(avg_s * max(self.total - self.n_done, 0))))
        w = len(str(self.total))
        print(
            f"  [{self.n_done:>{w}}/{self.total}]"
            f"  ETA {eta_str}"
            f"  errors={self.n_errors}",
            flush=True,
        )


print("Helpers defined OK.")

In [ ]:
def run_multi_turn(agent: AACAgent, row: "pd.Series", input_type: str) -> list[dict]:
    """Esegue la sequenza multi-turn per una riga e un input_type.

    PROPAGAZIONE DEL CONTESTO:
    - Turn 0: input reale (clear o vague) + EvalContext PIENO con mock tools.
      Se il planner decide call_tools=True (atteso su vague), _collect_context()
      usa i mock → time_of_day e schedule entrano nella SessionMemory via Turn.
    - Turn 1+: input="" + EvalContext VUOTO.
      Il planner legge prompt_summary() che include time_of_day e i topics
      del gold iniettato dal teacher forcing del turno precedente.
      Non chiama tool: il contesto è già in sessione.

    Teacher forcing: dopo ogni turno il gold pictogram viene iniettato nella
    memoria della sessione così il turno successivo parte dal gold precedente.
    """
    sentence = row["sentence"]
    concepts = row["concepts"]
    n_turns  = len(concepts)

    if input_type == "clear":
        caregiver_input_t0 = str(row["caregiver_clear"])
    else:
        caregiver_input_t0 = str(row["caregiver_vague"])

    agent.reset_session()
    results: list[dict] = []

    for turn_pos, concept_entry in enumerate(concepts):
        concept_text = concept_entry["concept_text"]
        gold_id      = int(concept_entry["gold_id"])

        if turn_pos == 0:
            ec        = build_eval_ctx(row)   # mock PIENO
            raw_input = caregiver_input_t0
        else:
            ec        = EvalContext()          # mock VUOTO — contesto già in sessione
            raw_input = ""

        window = agent.run(raw_input, eval_ctx=ec)

        # Raccogli dati PRIMA del teacher forcing
        predicted_ids    = [p.id for p in window]
        planner_concepts = [e["concept"] for e in agent.last_resolve_info]
        resolve_method   = next(
            (e["method"] for e in agent.last_resolve_info
             if e["concept"] == concept_text),
            "none"
        )

        results.append({
            "row_idx":             row.name,
            "input_type":          input_type,
            "turn_pos":            turn_pos,
            "concept_text":        concept_text,
            "called_get_time":     "get_time"     in ec.tool_calls,
            "called_get_schedule": "get_schedule" in ec.tool_calls,
            "predicted_ids":       str(predicted_ids),
            "plan_method":         agent.last_plan_method,
            "resolve_method":      resolve_method,
            "planner_concepts":    str(planner_concepts),
        })

        # Teacher forcing: inietta gold in memoria → il turno successivo
        # vede il gold nel prompt_summary e ragiona sul contesto corretto
        teacher_force(agent, gold_id, concept_text)

    return results


print("run_multi_turn defined OK.")

In [ ]:
LOG_EVERY  = 10
SAVE_EVERY = 10

csv_writer = IncrementalCSV(OUTPUT_CSV_PATH)
total_t0   = time.monotonic()

for model_name in MODELS:
    print(f"\n{'━'*70}\n  MODEL: {model_name}\n{'━'*70}")

    agent = HFAACAgent(
        model             = model_name,
        hf_device         = HF_DEVICE,
        hf_load_in_8bit   = LOAD_8BIT,
        hf_max_new_tokens = MAX_NEW_TOKENS,
        lang              = LANG_CODE,
        max_results       = EVAL_MAX_RESULTS,
    )

    tracker = ProgressTracker(total=len(df))

    for _, row in df.iterrows():
        split_val = str(row["split"])

        # Determina input_types da eseguire per questa riga
        if split_val == "none":
            continue
        elif split_val == "clear":
            input_types = ["clear"]
        elif split_val == "vague":
            input_types = ["vague"]
        else:  # "both"
            input_types = ["clear", "vague"]

        # Applica filtro globale NB_SPLIT_FILTER
        if SPLIT_FILTER != "all":
            input_types = [t for t in input_types if t == SPLIT_FILTER]
        if not input_types:
            continue

        try:
            for input_type in input_types:
                row_results = run_multi_turn(agent, row, input_type)
                csv_writer.add(row_results)
            tracker.record(row_results)  # conta la riga una volta sola
        except Exception as exc:
            tracker.record_error()
            print(f"  [ERROR] row={row.name}: {exc}", flush=True)

        if tracker.n_done % SAVE_EVERY == 0:
            csv_writer.flush()
        if tracker.n_done % LOG_EVERY == 0 or tracker.n_done == len(df):
            tracker.print_progress()

    csv_writer.flush()
    agent.unload()
    print(f"  Model {model_name!r} done.")

elapsed = time.monotonic() - total_t0
print(f"\nAll done in {timedelta(seconds=int(elapsed))}. Output: {OUTPUT_CSV_PATH}")

In [ ]:
res = pd.read_csv(OUTPUT_CSV_PATH)
for col in ("predicted_ids", "planner_concepts"):
    res[col] = res[col].apply(ast.literal_eval)

print(f"Righe totali          : {len(res)}")
print(f"\nDistribuzione input_type:\n{res['input_type'].value_counts()}")

t0 = res[res["turn_pos"] == 0]
print(f"\nTool call rate a turn_pos==0 (per input_type):")
print(t0.groupby("input_type")[["called_get_time","called_get_schedule"]].mean())
print("ATTESO: ~1.0 per vague, ~0.0 per clear")

print(f"\nplan_method distribution:\n{res['plan_method'].value_counts()}")
print(f"\nresolve_method distribution:\n{res['resolve_method'].value_counts()}")
print(f"\nDimensione finestra media: {res['predicted_ids'].apply(len).mean():.1f}")
print(f"\nEsempio prime 3 righe:\n{res.head(3).to_string()}")